In [ ]:
# ============================================================
# DEFINICIONES DE ESTADOS – PLEGADORA PLE-7
# ============================================================
#
# APAGADO:
#   - Representa apagado real de la máquina (corriente ≈ 0)
#   - Condición:
#       estado_R == 2
#       estado_S == 1
#       estado_T == 1
#   - Uso:
#       * Corte duro de eventos
#       * Delimitación de inicio y fin de día
#       * Identificación de paradas reales
#
# REPOSO OPERATIVO:
#   - Máquina encendida, sin trabajo efectivo
#   - Condición:
#       estado_R == 0
#       estado_S == 1
#       estado_T == 1
#   - Uso:
#       * Punto lógico previo al inicio de un evento
#       * Cierre normal de eventos
#       * Separación entre operaciones consecutivas
#
# TRABAJO EFECTIVO:
#   - Trabajo real de la plegadora
#   - Condición:
#       estado_R ∈ {1, 2}
#       estado_S ∈ {1, 2}
#       estado_T == 2
#       (se considera trabajo si aparece actividad en alguna fase)
#   - Uso:
#       * Inicio de eventos
#       * Continuidad del evento
#
# ============================================================
# PARÁMETROS CONFIGURABLES – DETECCIÓN DE EVENTOS (PLE-7)
# ============================================================
#
# N_ESTABLE:
#   - Cantidad mínima de muestras consecutivas en estado estable
#     (reposo_operativo o apagado)
#   - Uso:
#       * Determinar comienzo real del día
#       * Determinar fin real del día
#   - Valores típicos:
#       * Muestreo 1 s  → 60–120
#
# PAUSA_MIN:
#   - Cantidad mínima de reposo_operativo consecutivo
#   - Uso:
#       * Cierre normal de eventos
#       * Evitar fragmentación de un mismo ciclo
#   - Valores típicos:
#       * 30–60 s
#
# VENTANA_DESCARTE_APAGADO:
#   - Tiempo posterior al evento durante el cual, si el estado es
#     apagado sostenido, el evento se descarta
#   - Uso:
#       * Eliminación de eventos espurios de fin de jornada
#   - Valores típicos:
#       * 120–300 s
#
# VENTANA_PICO_INICIO (opcional):
#   - Ventana hacia atrás para ajustar el inicio del evento al último
#     pico eléctrico previo
#   - Uso:
#       * Refinamiento temporal del timestamp de inicio
#       * No afecta la lógica de detección
#
# ============================================================
# NOTA GENERAL:
#   - PLE-7 es más sensible a ruido eléctrico que PLE-1
#   - La definición de TRABAJO es más permisiva
#   - Si aparecen falsos eventos:
#       * Revisar primero PAUSA_MIN
#       * Luego definición de TRABAJO
# ============================================================


In [1]:
# Librerias

import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

In [2]:
# Leer rutas

from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "data"

#print("PROJECT_ROOT:", PROJECT_ROOT)
#print("DATA_PATH:", DATA_PATH)
#print("Existe data:", DATA_PATH.exists())

In [3]:
# Abrir los graficos en el navegador

pio.renderers.default = 'browser'

In [4]:
# Funcion para cargar datos

from pathlib import Path
import pandas as pd

def cargar_maquina(base_path, maquina):
    base_path = Path(base_path).resolve()
    paths = list((base_path / maquina).rglob("*.csv"))

    if not paths:
        raise ValueError(f"No se encontraron CSV para {maquina} en {base_path}")

    dfs = []
    for path in paths:
        df = pd.read_csv(path)
        df["maquina"] = maquina
        dfs.append(df)

    return (
        pd.concat(dfs, ignore_index=True)
          .sort_values("temporal_placa")
          .reset_index(drop=True)
    )

In [5]:
# Carga de datos

df_ple7 = cargar_maquina(DATA_PATH, "ple7")

In [6]:
# Funcion preparar_df

def preparar_df(df):
    df = df.copy()

    # Timestamp
    df['temporal_placa'] = pd.to_datetime(df['temporal_placa'])
    df['hora'] = df['temporal_placa'].dt.hour
    df['minuto'] = df['temporal_placa'].dt.minute

    # Turnos
    def asignar_turno(hora, minuto):
        t = hora * 60 + minuto

        # Pausas
        if 12*60 <= t < 12*60 + 30:
            return 'ALMUERZO'
        if 22*60 <= t < 22*60 + 30:
            return 'CENA'

        # Turnos
        if 5*60 <= t < 17*60:
            return 'TURNO MAÑANA'
        if 17*60 <= t < 22*60:
            return 'TURNO TARDE'
        if (t >= 22*60 + 30) or (t < 1*60):
            return 'TURNO TARDE'

        return 'FUERA_TURNO'

    df['turno'] = df.apply(
        lambda x: asignar_turno(x['hora'], x['minuto']),
        axis=1
    )
        
    # Potencias totales por timestamp
    df['p_activa_total'] = (
        df['potencia_a_r'] +
        df['potencia_a_s'] +
        df['potencia_a_t']
    )

    df['q_reactiva_total'] = (
        df['potencia_r_r'] +
        df['potencia_r_s'] +
        df['potencia_r_t']
    )

    return df


In [7]:
# Ajuste de datos de ple7

df_ple7 = preparar_df(df_ple7)

df_ple7['temporal_placa'] = (
    pd.to_datetime(df_ple7['temporal_placa'], errors='coerce')
    .dt.tz_localize(None)
)


In [8]:
# Ventanas para ple7

import numpy as np
import pandas as pd

win = 5  # muestras (si tu muestreo es ~1 Hz, equivale a 5 s)

rows = []

df_ple7['temporal_placa'] = pd.to_datetime(df_ple7['temporal_placa'], errors='coerce').dt.tz_localize(None)
df_ple7 = df_ple7.reset_index(drop=True)

fases = {
    'R': {'I': 'corriente_r', 'P': 'potencia_a_r'},
    'S': {'I': 'corriente_s', 'P': 'potencia_a_s'},
    'T': {'I': 'corriente_t', 'P': 'potencia_a_t'},
}

for fase, cols in fases.items():
    col_I = cols['I']
    col_P = cols['P']

    for i in range(0, len(df_ple7) - win, win):
        segI = df_ple7.iloc[i:i+win][col_I].astype(float).values
        segP = df_ple7.iloc[i:i+win][col_P].astype(float).values
        segT = df_ple7.iloc[i:i+win]['temporal_placa']

        # dt por muestra (robusto ante muestreo irregular)
        dt = segT.diff().dt.total_seconds().fillna(0).values

        # energía de ventana basada en potencia activa (J si P en W)
        energia = np.sum(segP * dt)

        rows.append({
            't_inicio': df_ple7.iloc[i]['temporal_placa'],
            'fase': fase,
            'media': np.mean(segI),
            'std': np.std(segI),
            'pendiente': segI[-1] - segI[0],
            'energia': energia
        })

df_feat = pd.DataFrame(rows)

df_R = df_feat[df_feat['fase'] == 'R'].reset_index(drop=True)
df_S = df_feat[df_feat['fase'] == 'S'].reset_index(drop=True)
df_T = df_feat[df_feat['fase'] == 'T'].reset_index(drop=True)


In [9]:
# Normalizacion por fase

from sklearn.preprocessing import StandardScaler

features = ['media', 'std', 'pendiente', 'energia']

scaler_R = StandardScaler()
Xn_R = scaler_R.fit_transform(df_R[features])

scaler_S = StandardScaler()
Xn_S = scaler_S.fit_transform(df_S[features])

scaler_T = StandardScaler()
Xn_T = scaler_T.fit_transform(df_T[features])



In [10]:
# Clustering por fase

from sklearn.cluster import KMeans

k_R = KMeans(n_clusters=4, random_state=42)
df_R['estado'] = k_R.fit_predict(Xn_R)

k_S = KMeans(n_clusters=4, random_state=42)
df_S['estado'] = k_S.fit_predict(Xn_S)

k_T = KMeans(n_clusters=4, random_state=42)
df_T['estado'] = k_T.fit_predict(Xn_T)



In [11]:
# Estados Fase R ple7

df_R.groupby('estado')[features].mean()


,media,std,pendiente,energia
estado,,,,
0,11.621885,0.090306,0.020700,3759.599174
1,14.968720,4.938083,-5.274212,12849.059372
2,0.315077,0.012975,0.013741,240.321418
3,16.439338,6.565097,10.267008,21053.046291


In [12]:
# Estados Fase S ple7

df_S.groupby('estado')[features].mean()


,media,std,pendiente,energia
estado,,,,
0,24.367536,0.209050,0.047426,16308.839010
1,0.314181,0.027476,0.002901,528.351053
2,32.498982,12.058654,-12.966456,38691.412120
3,35.943259,15.782050,24.520104,59535.323829


In [14]:
# Estados Fase T ple7

df_T.groupby('estado')[features].mean()


,media,std,pendiente,energia
estado,,,,
0,27.413078,0.231183,0.040972,58943.820732
1,0.500260,0.044099,-0.015101,400.072433
2,27.751193,0.164648,0.042397,30009.879085
3,36.144084,12.899636,-1.680535,73980.762639


In [35]:
# Nombramiento de estados

df_states = (
    df_R[['t_inicio', 'estado']]
    .rename(columns={'estado': 'estado_R'})
    .merge(
        df_S[['t_inicio', 'estado']].rename(columns={'estado': 'estado_S'}),
        on='t_inicio'
    )
    .merge(
        df_T[['t_inicio', 'estado']].rename(columns={'estado': 'estado_T'}),
        on='t_inicio'
    )
)

# -----------------------------
# Apagado real (corriente ~ 0)
# -----------------------------
df_states['apagado'] = (
    (df_states['estado_R'] == 2) &
    (df_states['estado_S'] == 1) &
    (df_states['estado_T'] == 1)
)

# -----------------------------
# Reposo operativo (encendida sin trabajo)
# -----------------------------
df_states['reposo_operativo'] = (
    (df_states['estado_R'] == 0) &
    (df_states['estado_S'] == 0) &
    (df_states['estado_T'].isin([0, 2]))
)

# -----------------------------
# Trabajo efectivo
# -----------------------------
df_states['trabajo'] = (
    (df_states['estado_R'].isin([1, 3])) |
    (df_states['estado_S'].isin([2, 3])) |
    (df_states['estado_T'] == 3)
)




In [28]:
PAUSA_MIN = 3      # ventanas consecutivas de reposo para cerrar evento
N_ESTABLE = 3      # ventanas consecutivas de reposo para validar zona
umbral_pico_ple7 = 0.4
VENTANA_ANTI_APAGADO = 180  # segundos


In [36]:
# Determinar comienzo y fin reales de los datos

# -------- INICIO VÁLIDO --------
contador = 0
idx_inicio_valido = None

for i, row in df_states.iterrows():
    estado_estable = row['reposo_operativo'] or row['apagado']
    if estado_estable:
        contador += 1
        if contador >= N_ESTABLE:
            idx_inicio_valido = i
            break
    else:
        contador = 0

# -------- FIN VÁLIDO --------
contador_no_trabajo = 0
idx_fin_valido = None

for i in range(len(df_states) - 1, -1, -1):
    row = df_states.loc[i]

    if not row['trabajo']:
        contador_no_trabajo += 1
    else:
        if contador_no_trabajo >= PAUSA_MIN:
            idx_fin_valido = i
            break
        contador_no_trabajo = 0

# -------- Fallbacks --------
if idx_inicio_valido is None:
    idx_inicio_valido = df_states.index.min()

if idx_fin_valido is None:
    idx_fin_valido = df_states.index.max()

if idx_inicio_valido >= idx_fin_valido:
    raise ValueError("No se pudo determinar una zona válida de operación")

t_inicio_valido = df_states.loc[idx_inicio_valido, 't_inicio']
t_fin_valido    = df_states.loc[idx_fin_valido, 't_inicio']


In [37]:
# Funcion detectar_picos mediante corriente

def detectar_picos_i(signal, timestamp, umbral_pico, ventana):

    dt = timestamp.diff().dt.total_seconds()
    valid = dt > 0

    dI_dt = signal.diff() / dt

    # subida fuerte
    subida = valid & dI_dt.notna() & (dI_dt > umbral_pico)

    picos = subida.copy() * False

    # buscar máximo local después de la subida
    for i in range(1, len(signal) - ventana):
        if subida.iloc[i]:
            entorno = signal.iloc[i:i+ventana+1]
            if signal.iloc[i+1] == entorno.max():
                picos.iloc[i+1] = True

    return picos


In [38]:
# Detección de picos en las 3 fases de la ple7

import pandas as pd

# -----------------------------
# Deteccion de picos en corriente por fase
# -----------------------------
mask_picos_Ir = detectar_picos_i(
    df_ple7['corriente_r'],
    df_ple7['temporal_placa'],
    umbral_pico = umbral_pico_ple7,
    ventana=2
)

mask_picos_Is = detectar_picos_i(
    df_ple7['corriente_s'],
    df_ple7['temporal_placa'],
    umbral_pico = umbral_pico_ple7,
    ventana=2
)

mask_picos_It = detectar_picos_i(
    df_ple7['corriente_t'],
    df_ple7['temporal_placa'],
    umbral_pico = umbral_pico_ple7,
    ventana=2
)

df_ple7['pico_Ir'] = mask_picos_Ir
df_ple7['pico_Is'] = mask_picos_Is
df_ple7['pico_It'] = mask_picos_It

# -----------------------------
# DataFrame de picos
# -----------------------------
mask_picos_final = (
    df_ple7['pico_Ir'] |
    df_ple7['pico_Is'] |
    df_ple7['pico_It']
)

df_peaks_all_ple7 = (
    df_ple7.loc[
        mask_picos_final,
        ['temporal_placa', 'pico_Ir', 'pico_Is', 'pico_It']
    ]
    .rename(columns={'temporal_placa': 't'})
    .reset_index(drop=True)
)

def origen_pico(row):
    origenes = []
    if row['pico_Ir']:
        origenes.append('Ir')
    if row['pico_Is']:
        origenes.append('Is')
    if row['pico_It']:
        origenes.append('It')
    return '+'.join(origenes)

df_peaks_all_ple7['origen'] = df_peaks_all_ple7.apply(origen_pico, axis=1)

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
df_peaks_all_ple7

,t,pico_Ir,pico_Is,pico_It,origen
0,2026-01-09 17:26:37,False,False,True,It
1,2026-01-09 17:51:32,False,False,True,It
2,2026-01-09 17:54:54,True,True,True,Ir+Is+It
3,2026-01-09 18:03:47,True,True,True,Ir+Is+It
4,2026-01-09 18:13:36,True,True,True,Ir+Is+It
5,2026-01-09 19:12:12,False,True,True,Is+It
6,2026-01-09 19:12:52,True,True,True,Ir+Is+It
7,2026-01-09 19:55:54,True,True,True,Ir+Is+It
8,2026-01-09 20:12:04,False,True,True,Is+It
9,2026-01-09 20:12:14,True,True,True,Ir+Is+It


In [42]:
# Definicion y determinacion de EVENTOS

eventos = []

en_evento = False
inicio = None
contador_reposo = 0
ultimo_reposo = None

for i in range(idx_inicio_valido, idx_fin_valido + 1):

    row = df_states.loc[i]

    apagado = row['apagado']
    reposo = row['reposo_operativo']
    trabajo = row['trabajo']

    # -----------------------------
    # Apagado: cortar evento
    # -----------------------------
    if apagado:
        if en_evento:
            eventos.append({
                'Fecha Inicio': inicio,
                'Fecha Fin': row['t_inicio']
            })
            en_evento = False
            inicio = None
        contador_reposo = 0
        ultimo_reposo = None
        continue

    # -----------------------------
    # Guardar último reposo
    # -----------------------------
    if reposo and not en_evento:
        ultimo_reposo = row['t_inicio']

    # -----------------------------
    # INICIO DE EVENTO
    # -----------------------------
    if trabajo and not en_evento:

        # BLOQUE ANTI-APAGADO
        t_actual = row['t_inicio']
        t_min = t_actual - pd.Timedelta(seconds=VENTANA_ANTI_APAGADO)

        hubo_apagado_reciente = df_states.loc[
            (df_states['t_inicio'] >= t_min) &
            (df_states['t_inicio'] < t_actual),
            'apagado'
        ].any()

        if hubo_apagado_reciente:
            continue  # ignorar falso evento

        inicio = ultimo_reposo if ultimo_reposo is not None else t_actual
        en_evento = True
        contador_reposo = 0

    # -----------------------------
    # CONTINUIDAD
    # -----------------------------
    if en_evento and trabajo:
        contador_reposo = 0

    # -----------------------------
    # CIERRE NORMAL
    # -----------------------------
    if en_evento and reposo:
        contador_reposo += 1
        if contador_reposo >= PAUSA_MIN:
            fin = df_states.loc[i - PAUSA_MIN, 't_inicio']
            eventos.append({
                'Fecha Inicio': inicio,
                'Fecha Fin': fin
            })
            en_evento = False
            inicio = None
            contador_reposo = 0
            ultimo_reposo = row['t_inicio']

# -------- cerrar evento abierto --------
if en_evento:
    eventos.append({
        'Fecha Inicio': inicio,
        'Fecha Fin': df_states.loc[idx_fin_valido, 't_inicio']
    })

df_eventos = pd.DataFrame(eventos)



In [43]:
# Filtrado de picos durante apagado

VENTANA_DESCARTE_APAGADO = 180  # segundos

eventos_filtrados = []

for _, ev in df_eventos.iterrows():

    t_fin = ev['Fecha Fin']
    t_limite = t_fin + pd.Timedelta(seconds=VENTANA_DESCARTE_APAGADO)

    ventana_post = df_states[
        (df_states['t_inicio'] > t_fin) &
        (df_states['t_inicio'] <= t_limite)
    ]

    # Si no hay datos suficientes, conservar evento
    if ventana_post.empty:
        eventos_filtrados.append(ev)
        continue

    # Verificar si TODO el intervalo está en apagado
    apagado_sostenido = ventana_post['apagado'].all()

    if not apagado_sostenido:
        eventos_filtrados.append(ev)

df_eventos = pd.DataFrame(eventos_filtrados).reset_index(drop=True)


In [44]:
# Correccion del comienzo de los eventos mediante la deteccion de picos

VENTANA_PICO_INICIO = 5  # segundos

df_eventos = df_eventos.copy()
df_eventos['Fecha Inicio Ajustada'] = df_eventos['Fecha Inicio']

for i, ev in df_eventos.iterrows():

    t_ini = ev['Fecha Inicio']
    t_min = t_ini - pd.Timedelta(seconds=VENTANA_PICO_INICIO)

    # picos cercanos antes del evento
    picos_previos = df_peaks_all_ple7[
        (df_peaks_all_ple7['t'] >= t_min) &
        (df_peaks_all_ple7['t'] < t_ini)
    ]

    if not picos_previos.empty:
        # último pico previo como inicio real
        df_eventos.at[i, 'Fecha Inicio Ajustada'] = picos_previos['t'].max()


In [45]:
# Graficas de eventos para ple7

import plotly.graph_objects as go

maq = 'ple7'

map_fase_col = {
    'R': 'corriente_r',
    'S': 'corriente_s',
    'T': 'corriente_t'
}

colores_fase = {
    'R': 'red',
    'S': 'green',
    'T': 'blue'
}

fig = go.Figure()
trace_idx = {}

# Trazas de corriente
for fase, col in map_fase_col.items():

    visible = (fase == 'R')

    fig.add_trace(
        go.Scatter(
            x=df_ple7['temporal_placa'],
            y=df_ple7[col],
            mode='lines',
            name=f'Fase {fase}',
            line=dict(color=colores_fase[fase]),
            visible=visible
        )
    )

    trace_idx[fase] = len(fig.data) - 1

# Shapes de eventos (comunes a todas las fases)
shapes_eventos = []

for _, ev in df_eventos.iterrows():
    shapes_eventos.append(
        dict(
            type='rect',
            xref='x',
            yref='paper',
            x0=ev['Fecha Inicio Ajustada'],
            x1=ev['Fecha Fin'],
            y0=0,
            y1=1,
            fillcolor='grey',
            opacity=0.2,
            line_width=0
        )
    )

# Botones por fase
botones = []

for fase in map_fase_col.keys():

    visibles = [False] * len(fig.data)
    visibles[trace_idx[fase]] = True

    botones.append(
        dict(
            label=f'Fase {fase}',
            method='update',
            args=[
                {'visible': visibles},
                {'shapes': shapes_eventos}
            ]
        )
    )

fig.update_layout(
    title='ple7 – Corriente por fase con eventos de máquina',
    xaxis_title='Tiempo',
    yaxis_title='Corriente [A]',
    template='plotly_white',
    shapes=shapes_eventos,
    updatemenus=[
        dict(
            buttons=botones,
            direction='down',
            x=1.08,
            y=1.1,
            showactive=True
        )
    ]
)

fig.show()


In [ ]:
# Eventos a .cvs
from pathlib import Path

OUT_TABLAS = PROJECT_ROOT / "output" / "tablas"
OUT_TABLAS.mkdir(parents=True, exist_ok=True)

df_eventos.to_csv(OUT_TABLAS / "df_eventos_ple7.csv", index=False)